## swiGLU 4 - Retraining (global recovery) Analysis

The purpose of this notebook is to test different model retraining configurations and approaches on a smaller retraining (global reocvery) budgets.

Final winning configuration and approach will be then used for production retraining experiments.

setup

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'src' / 'mlp_replacement').is_dir():
            return candidate
    raise RuntimeError('Could not locate the repository root')


PROJECT_ROOT = find_project_root()
ARTIFACT_PATH = None  # Set an explicit completed swiglu-4 JSON path when needed.
if ARTIFACT_PATH is None:
    search_roots = [
        PROJECT_ROOT / 'data/results/workflows/model/swiglu-4',
        PROJECT_ROOT / 'data/results/workflows/models/swiglu-4',
    ]
    candidates = sorted(
        (path for root in search_roots if root.is_dir() for path in root.glob('*.json')
         if not path.name.endswith('.run.json')),
        key=lambda path: path.stat().st_mtime,
    )
    if not candidates:
        raise FileNotFoundError('No swiglu-4 science artifact was found')
    ARTIFACT_PATH = candidates[-1]
else:
    ARTIFACT_PATH = Path(ARTIFACT_PATH)
    if not ARTIFACT_PATH.is_absolute():
        ARTIFACT_PATH = PROJECT_ROOT / ARTIFACT_PATH

artifact = json.loads(ARTIFACT_PATH.read_text(encoding='utf-8'))
if artifact.get('workflow') != 'swiglu-4' or artifact.get('schema_version') != 1:
    raise ValueError('Expected a schema-1 swiglu-4 artifact')
if artifact.get('status') != 'completed':
    raise ValueError(f"Artifact is not completed: {artifact.get('status')}")

configuration = artifact['configuration']
results = artifact['results']
print(ARTIFACT_PATH.relative_to(PROJECT_ROOT))
display(pd.DataFrame([{
    'execution_mode': configuration['execution_mode'],
    'sparsity_key': results['source_state']['sparsity_key'],
    'calibration_pairs': results['source_state']['calibration_pairs'],
    'requested_target_tokens': configuration['recovery']['target_tokens'],
    'effective_batch_tokens': results['data']['effective_batch_tokens'],
    'dense_wikitext_ppl': results['dense_baseline']['wikitext_validation']['perplexity'],
}]))

#### Configuration testing
- learning rate, scheduling
- KL, KL+CE
- weight decay

In [ ]:
configuration_stage = results['configuration_testing']
configuration_trajectories = configuration_stage['trajectories']
candidate_rows = []
for trajectory_id, trajectory in configuration_trajectories.items():
    values = trajectory['configuration']
    candidate_rows.append({
        'id': trajectory_id,
        'label': trajectory['label'],
        'lr': values['learning_rate'],
        'scheduler': values['scheduler'],
        'warmup_fraction': values['warmup_fraction'],
        'temperature': values['temperature'],
        'ce_weight': values['ce_weight'],
        'weight_decay': values['weight_decay'],
        'tokens_seen': trajectory['tokens_seen'],
        'trainable_parameters': trajectory['trainable_scope']['trainable_parameters'],
    })
display(pd.DataFrame(candidate_rows).sort_values('id').reset_index(drop=True))
display(pd.DataFrame([configuration_stage['optimizer_selection']]))
display(pd.DataFrame([configuration_stage['objective_selection']]))
display(pd.DataFrame([configuration_stage['winner']]))

configuration_history = pd.DataFrame([
    {'id': trajectory_id, **row}
    for trajectory_id, trajectory in configuration_trajectories.items()
    for row in trajectory['validation_history']
])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for trajectory_id, group in configuration_history.groupby('id'):
    group = group.sort_values('tokens_seen')
    axes[0].plot(group['tokens_seen'] / 1e6, group['recovery_validation_kl'], marker='.', label=trajectory_id)
    lr = group['learning_rates'].map(lambda values: next(iter(values.values())) if values else None)
    axes[1].plot(group['tokens_seen'] / 1e6, lr, marker='.', label=trajectory_id)
axes[0].set(xlabel='Actual recovery tokens (M)', ylabel='Fixed validation KL (T=1)', title='Configuration recovery')
axes[1].set(xlabel='Actual recovery tokens (M)', ylabel='Replacement learning rate', title='Learning-rate histories')
axes[1].set_yscale('log')
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()
plt.show()

configuration_milestones = pd.DataFrame([
    {
        'id': trajectory_id,
        'requested_tokens': row['requested_tokens'][0],
        'actual_tokens': row['actual_tokens'],
        'validation_kl_t1': row['current']['recovery_validation_kl_t1'],
        'wikitext_ppl': row['current']['wikitext_validation']['perplexity'],
        'allocation_ppl': row['current']['allocation_selection']['perplexity'],
    }
    for trajectory_id, trajectory in configuration_trajectories.items()
    for row in trajectory['milestones']
])
display(configuration_milestones.sort_values(['requested_tokens', 'validation_kl_t1']).reset_index(drop=True))

#### RMSNorm

In [ ]:
rms_stage = results['rmsnorm']
rms_trajectories = rms_stage['trajectories']
display(pd.DataFrame([rms_stage['selection']]))
rms_rows = []
for trajectory_id, trajectory in rms_trajectories.items():
    endpoint = trajectory['endpoint']
    rms_rows.append({
        'id': trajectory_id,
        'label': trajectory['label'],
        'rmsnorm_lr_multiplier': trajectory['configuration']['rmsnorm_learning_rate_multiplier'],
        'actual_tokens': endpoint['actual_tokens'],
        'validation_kl_t1': endpoint['recovery_validation_kl_t1'],
        'wikitext_ppl': endpoint['metrics']['wikitext_validation']['perplexity'],
        'trainable_parameters': trajectory['trainable_scope']['trainable_parameters'],
    })
display(pd.DataFrame(rms_rows).sort_values('id').reset_index(drop=True))

fig, axis = plt.subplots(figsize=(7, 4))
for trajectory_id, trajectory in rms_trajectories.items():
    history = pd.DataFrame(trajectory['validation_history']).sort_values('tokens_seen')
    axis.plot(history['tokens_seen'] / 1e6, history['recovery_validation_kl'], marker='.', label=trajectory_id)
axis.set(xlabel='Actual recovery tokens (M)', ylabel='Fixed validation KL (T=1)', title='RMSNorm scope comparison')
axis.grid(alpha=0.25)
axis.legend()
plt.show()

rms_groups = pd.DataFrame([
    {'trajectory': trajectory_id, **{key: value for key, value in group.items() if key != 'parameter_names'}}
    for trajectory_id, trajectory in rms_trajectories.items()
    for group in trajectory['trainable_scope']['parameter_groups']
])
display(rms_groups)

#### LoRA

mlp subset

In [ ]:
lora_stage = results['lora']
l1 = lora_stage['trajectories']['L1']
display(pd.DataFrame([{
    'id': 'L1',
    'scope': l1['configuration']['variant'],
    'rank': l1['configuration']['lora_rank'],
    'alpha': l1['configuration']['lora_alpha'],
    'adapter_lr': l1['configuration']['lora_learning_rate'],
    'target_modules': len(l1['trainable_scope']['lora_paths']),
    'trainable_parameters': l1['trainable_scope']['trainable_parameters'],
    'validation_kl_t1': l1['endpoint']['recovery_validation_kl_t1'],
    'wikitext_ppl': l1['endpoint']['metrics']['wikitext_validation']['perplexity'],
}]))
l1_history = pd.DataFrame(l1['validation_history']).sort_values('tokens_seen')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(l1_history['tokens_seen'] / 1e6, l1_history['recovery_validation_kl'], marker='.')
axes[0].set(xlabel='Actual recovery tokens (M)', ylabel='Fixed validation KL (T=1)', title='L1 validation')
axes[1].plot(l1_history['tokens_seen'] / 1e6, l1_history['mean_train_kl_since_resume'], label='KL')
axes[1].plot(l1_history['tokens_seen'] / 1e6, l1_history['mean_train_loss_since_resume'], label='total')
axes[1].set(xlabel='Actual recovery tokens (M)', ylabel='Training loss', title='L1 training history')
for axis in axes:
    axis.grid(alpha=0.25)
axes[1].legend()
plt.tight_layout()
plt.show()

entire model

In [ ]:
l2 = lora_stage['trajectories']['L2']
display(pd.DataFrame([{
    'id': 'L2',
    'scope': l2['configuration']['variant'],
    'rank': l2['configuration']['lora_rank'],
    'alpha': l2['configuration']['lora_alpha'],
    'adapter_lr': l2['configuration']['lora_learning_rate'],
    'target_modules': len(l2['trainable_scope']['lora_paths']),
    'trainable_parameters': l2['trainable_scope']['trainable_parameters'],
    'validation_kl_t1': l2['endpoint']['recovery_validation_kl_t1'],
    'wikitext_ppl': l2['endpoint']['metrics']['wikitext_validation']['perplexity'],
    'embeddings_lm_head_tied': l2['trainable_scope']['embedding_and_lm_head']['weight_storage_tied'],
}]))
display(pd.DataFrame([{
    'excluded_module': module,
    'reason': 'tied vocabulary table is outside the transformer-recovery scope',
} for module in l2['trainable_scope']['excluded_modules']]))
display(pd.DataFrame([lora_stage['comparison']]))

fig, axis = plt.subplots(figsize=(7, 4))
for trajectory_id in ('L1', 'L2'):
    history = pd.DataFrame(lora_stage['trajectories'][trajectory_id]['validation_history']).sort_values('tokens_seen')
    axis.plot(history['tokens_seen'] / 1e6, history['recovery_validation_kl'], marker='.', label=trajectory_id)
axis.set(xlabel='Actual recovery tokens (M)', ylabel='Fixed validation KL (T=1)', title='LoRA scope comparison')
axis.grid(alpha=0.25)
axis.legend()
plt.show()

#### Resutls

winner comparison

In [ ]:
comparison = pd.DataFrame(results['final_comparison'])
display(comparison.sort_values('recovery_validation_kl_t1').reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ordered = comparison.sort_values('recovery_validation_kl_t1')
axes[0].bar(ordered['role'], ordered['recovery_validation_kl_t1'])
target_millions = configuration['recovery']['target_tokens'] / 1e6
axes[0].set(ylabel='Fixed validation KL (T=1)', title=f'{target_millions:g}M recovery comparison')
axes[0].tick_params(axis='x', rotation=55)
axes[1].bar(ordered['role'], ordered['wikitext_validation_perplexity'])
axes[1].axhline(results['dense_baseline']['wikitext_validation']['perplexity'], color='black', linestyle='--', label='dense')
axes[1].set(ylabel='WikiText-2 validation perplexity', title='Quality after recovery')
axes[1].tick_params(axis='x', rotation=55)
axes[1].legend()
for axis in axes:
    axis.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

selected_ids = {
    'C0': ('configuration_testing', 'C0'),
    'configuration winner': ('configuration_testing', results['configuration_testing']['winner']['id']),
    'RMSNorm winner': ('rmsnorm', results['rmsnorm']['winner']['id']),
    'L1': ('lora', 'L1'),
    'L2': ('lora', 'L2'),
}
fig, axis = plt.subplots(figsize=(8, 5))
for label, (section, trajectory_id) in selected_ids.items():
    trajectories = results[section]['trajectories']
    history = pd.DataFrame(trajectories[trajectory_id]['validation_history']).sort_values('tokens_seen')
    axis.plot(history['tokens_seen'] / 1e6, history['recovery_validation_kl'], marker='.', label=label)
axis.set(xlabel='Actual recovery tokens (M)', ylabel='Fixed validation KL (T=1)', title='Selected recovery histories')
axis.grid(alpha=0.25)
axis.legend()
plt.show()

In [ ]:
display(pd.DataFrame(results['runtime']))
display(pd.DataFrame([{
    'artifact_status': artifact['status'],
    'source_artifact': artifact['provenance']['source_paths']['swiglu_3_artifact'],
    'source_sha256': artifact['provenance']['source_sha256']['swiglu_3_artifact'],
    'packed_token_fingerprint': results['data']['packed_token_cache']['fingerprint'],
    'persistent_weight_assets': artifact['storage']['persistent_weight_assets'],
    'temporary_scratch_removed': artifact['storage']['temporary_scratch_removed'],
}]))
display(pd.DataFrame(results['data']['operator_state_files']).T.reset_index(names='layer'))